In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path('Telco_Customer_Churn_Dataset  (3).csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Task 1: Business Understanding

The goal is to predict customer churn in a telecom company and understand the drivers behind it.

In [2]:
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(df.columns.tolist())
print('\nChurn distribution:')
print(df['Churn'].value_counts(normalize=True))

Rows: 7043
Columns: 21
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Churn distribution:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


## Task 2: Data Quality Checks

We inspect missing values, fix numeric conversion issues, and check for duplicate customer records.

In [3]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

if missing.sum() == 0:
    print('No missing values found.')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Missing values after cleaning TotalCharges: {df['TotalCharges'].isnull().sum()}")

duplicate_customers = df.duplicated(subset=['customerID']).sum()
print(f'Duplicate customer IDs: {duplicate_customers}')
df = df.drop_duplicates(subset=['customerID']).reset_index(drop=True)
df.head()

Missing values per column:
Series([], dtype: int64)
No missing values found.
Missing values after cleaning TotalCharges: 11
Duplicate customer IDs: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Task 3: Exploratory Data Analysis

We examine the main churn patterns by contract type, internet service, and payment method.

In [4]:
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda s: s.eq('Yes').mean()).sort_values(ascending=False)
print('Churn rate by contract type:')
print(churn_by_contract)

churn_by_internet = df.groupby('InternetService')['Churn'].apply(lambda s: s.eq('Yes').mean()).sort_values(ascending=False)
print('\nChurn rate by internet service:')
print(churn_by_internet)

churn_by_payment = df.groupby('PaymentMethod')['Churn'].apply(lambda s: s.eq('Yes').mean()).sort_values(ascending=False)
print('\nChurn rate by payment method:')
print(churn_by_payment)

Churn rate by contract type:
Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Churn, dtype: float64

Churn rate by internet service:
InternetService
Fiber optic    0.418928
DSL            0.189591
No             0.074050
Name: Churn, dtype: float64

Churn rate by payment method:
PaymentMethod
Electronic check             0.452854
Mailed check                 0.191067
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Name: Churn, dtype: float64


In [5]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Churn', hue='Churn', palette={'No': 'steelblue', 'Yes': 'firebrick'})
plt.title('Churn Distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'churn_distribution.png', dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='Contract', y=df['Churn'].map({'No': 0, 'Yes': 1}), estimator='mean')
plt.title('Average Churn Rate by Contract Type')
plt.xlabel('Contract')
plt.ylabel('Churn Rate')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'contract_vs_churn.png', dpi=200)
plt.close()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', hue='Churn', dodge=False, palette={'No': 'steelblue', 'Yes': 'firebrick'}, legend=False)
plt.title('Monthly Charges by Churn Status')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'monthly_charges_by_churn.png', dpi=200)
plt.close()

print('EDA plots saved to outputs folder.')

EDA plots saved to outputs folder.


## Task 4: Preprocessing for Modeling

Prepare a model-ready dataset by removing identifiers, encoding categorical columns, and creating a feature matrix.

In [6]:
features = df.drop(columns=['customerID', 'Churn']).copy()
target = df['Churn'].map({'No': 0, 'Yes': 1}).astype(int)

object_cols = [
    col for col in features.columns
    if pd.api.types.is_object_dtype(features[col]) or pd.api.types.is_string_dtype(features[col])
]
for col in object_cols:
    features[col] = features[col].astype(str)

X = pd.get_dummies(features, drop_first=True)
y = target

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print('Preprocessed dataset is ready for modeling.')

Feature matrix shape: (7043, 30)
Target shape: (7043,)
Preprocessed dataset is ready for modeling.
